# Day 1 · Hands-on 4 — False positives (Type I errors)
**About 30 min · work in pairs** (one person drives the keyboard, the other reads, predicts and questions — swap halfway)


We split GBSG2 patients into a fake "Drug" group and a fake "Placebo" group **by coin flip**, then compare tumour size. Nobody received anything, so there is **no real effect**. Any "significant" result is a false positive.



> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to write, change, fix or explain *code*, including new simulations.
> - 🎓 **Claude / ChatGPT (other tab) = tutor.** Use it to explain ideas, quiz you and critique your own work. Ask for its reasoning, and check one claim yourself in the notebook.


**How this notebook works:** run cells from top to bottom. Each code cell has exactly one line marked `# 🔧 CHANGE THIS` — that is the only thing you need to edit. Lines starting with 📋 are results to enter in the shared sheet.

In [ ]:
# ▶️ Run this cell first (about 20 seconds). Nothing to change here.
%pip install -q lifelines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from lifelines.datasets import load_gbsg2

df = load_gbsg2()
print(f"✅ Loaded GBSG2: {len(df)} patients, {df.shape[1]} variables")

## A. 🧩 Puzzle (run this before the lecture)
Each row is one fake experiment with 50 patients per group. Use your **pair number** as the seed.

✍️ **Predict first:** out of 20 experiments with no real effect, how many will reach p < 0.05?

In [ ]:
# 🔧 CHANGE THIS: your pair number
seed = 1

rng = np.random.default_rng(seed)
tsize = df["tsize"].to_numpy()
rows = []
for experiment in range(1, 21):
    chosen = rng.choice(len(tsize), size=100, replace=False)   # 100 random patients
    drug, placebo = tsize[chosen[:50]], tsize[chosen[50:]]     # coin-flip groups
    p = stats.ttest_ind(drug, placebo, equal_var=False).pvalue
    rows.append({"experiment": experiment,
                 "difference (mm)": round(drug.mean() - placebo.mean(), 1),
                 "p-value": round(p, 3),
                 "significant?": "⚠️ YES" if p < 0.05 else ""})
results = pd.DataFrame(rows)
display(results)

n_sig = (results["p-value"] < 0.05).sum()
print("📋 For the shared sheet:")
print(f"   seed = {seed}   |   'significant' results = {n_sig} / 20")

## B. After the lecture: 2,000 experiments
### 🧭 Concept
With no real effect, how often do we get p < α? Look also at the **shape** of the p-value histogram. Try α = 0.05, 0.01 and 0.10.

In [ ]:
# 🔧 CHANGE THIS: significance threshold alpha
alpha = 0.05

n_experiments = 2000
rng = np.random.default_rng()
tsize = df["tsize"].to_numpy()

# Each row: 100 random patients, first 50 = "Drug", last 50 = "Placebo"
order = np.argsort(rng.random((n_experiments, len(tsize))), axis=1)[:, :100]
groups = tsize[order]
p_values = stats.ttest_ind(groups[:, :50], groups[:, 50:], axis=1, equal_var=False).pvalue

plt.figure(figsize=(7, 3.5))
plt.hist(p_values, bins=20, range=(0, 1), color="gray", edgecolor="white")
plt.axvline(alpha, color="red", linewidth=2, label=f"α = {alpha}")
plt.xlabel("p-value"); plt.ylabel("Number of experiments"); plt.legend()
plt.title("p-values from 2,000 experiments with NO real effect")
plt.show()
print(f"False-positive rate: {np.mean(p_values < alpha):.1%}   (alpha = {alpha:.0%})")

### ✅ Concept check
1. What shape does the p-value histogram have when there is no effect?
2. Why does the false-positive rate end up close to α, whatever α you choose?

<details><summary>Show answers</summary>

1. Flat (uniform): with no effect, every p-value between 0 and 1 is equally likely.
2. Because of that flatness, the fraction below any threshold α is about α. **α is the false-positive rate you agree to accept**, decided before you see the data.
</details>

## C. One study, many outcomes
### 🧭 Concept
Real studies rarely test one thing. Suppose a study measures several biomarkers and **none** is affected by treatment. What is the chance that at least one comes out "significant"? Try 1, 5, 20 and 100 tests. (An RNA-seq experiment tests about 20,000 genes, which is Day 2.)

In [ ]:
# 🔧 CHANGE THIS: number of independent tests in one study
n_tests = 20

alpha, n_studies, n = 0.05, 2000, 30
rng = np.random.default_rng()
# n_studies studies × n_tests biomarkers × n patients per group, no real effects anywhere
a = rng.normal(size=(n_studies, n_tests, n))
b = rng.normal(size=(n_studies, n_tests, n))
p = stats.ttest_ind(a, b, axis=2).pvalue
at_least_one = np.mean((p < alpha).any(axis=1))

print(f"{n_tests} tests per study, no real effects:")
print(f"   studies with at least one 'significant' result: {at_least_one:.0%}")
print(f"   theory, 1 - (1 - α)^k:                           {1 - (1 - alpha) ** n_tests:.0%}")

## 🤖 Go deeper with AI
Three ways to keep learning about multiple testing: now, if you finish early, or after the workshop.

**🎓 Explain and quiz me** (Claude / ChatGPT):
> *Explain the multiple-comparisons problem with a lab example. Then describe 3 short studies to me, one at a time, and ask whether each one needs a correction and why.*

**🧪 See it in a simulation** (🧮 Gemini in Colab, in a new cell at the end of this notebook):
> *Simulate 1,000 studies that each test 20 biomarkers with no real effect. Show the share of studies with at least one p < 0.05, with no correction, with Bonferroni and with Benjamini–Hochberg.*

**🔬 Apply it to my work** (Claude / ChatGPT; replace the brackets with your own details):
> *Here is everything I compared in my last experiment, including what I didn't report: [list]. How many tests did I really run, and how should I report the one that was significant?*

✔ **Check one claim yourself:** Without correction, about 64% of studies should have at least one false positive (1 − 0.95²⁰).

## 💬 Discuss with your partner
- The shared sheet pools every pair's 20 experiments. Was the overall rate close to 5%? Why did some pairs see 0 and others 3?
- Where in your own work do you run many tests on the same data (many genes, time points, subgroups, outcomes)?
- If you lower α to 0.01 to avoid false positives, what do you lose? (That's the next hands-on.)